In [ ]:
# ============================================
# 04_MobileNetV2.ipynb
# MOBILENETV2 - TRANSFER LEARNING + FINE-TUNING
# DETEKSI KANTUK PENGEMUDI
# ============================================

import os
import sys
import json
import time
import datetime
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# ============================================
# TENSORFLOW SETUP
# ============================================
warnings.filterwarnings('ignore')
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU

import tensorflow as tf

# CPU Optimization
tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(4)

from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping, 
    ReduceLROnPlateau, 
    ModelCheckpoint,
    TensorBoard
)
from tensorflow.keras.optimizers import Adam

print("="*60)
print("🚀 MOBILENETV2 - TRANSFER LEARNING + FINE-TUNING")
print("="*60)

# ============================================
# 1. SYSTEM INFO
# ============================================

print(f"\n[System Info]")
print(f"   - Python: {sys.version.split()[0]}")
print(f"   - TensorFlow: {tf.__version__}")
print(f"   - Mode: CPU (forced)")
print(f"   - CPU Threads: {tf.config.threading.get_intra_op_parallelism_threads()}")

# ============================================
# 2. SETUP PATHS
# ============================================

project_root = Path.cwd().parent if Path.cwd().name in ['notebooks', 'notebook'] else Path.cwd()
processed_dir = project_root / "dataset" / "processed" / "cnn_mobilenet"
model_dir = project_root / "models" / "mobilenetv2"
result_dir = project_root / "results" / "mobilenetv2"

model_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

print(f"\n[Paths]")
print(f"   - Project: {project_root}")
print(f"   - Data: {processed_dir}")
print(f"   - Model: {model_dir}")
print(f"   - Results: {result_dir}")

# ============================================
# 3. CONFIGURATION
# ============================================

CONFIG = {
    'img_size': (224, 224),  # MobileNetV2 default
    'batch_size': 16,
    'phase1_epochs': 15,     # Feature extraction (base frozen)
    'phase2_epochs': 15,     # Fine-tuning (unfreeze top layers)
    'learning_rate': 0.001,  # Phase 1
    'fine_tune_lr': 1e-5,    # Phase 2 (lebih kecil)
    'early_stopping_patience': 8,
    'reduce_lr_patience': 4,
    'class_names': ['awake', 'drowsy']
}

print("\n[Configuration]")
for key, value in CONFIG.items():
    print(f"   - {key}: {value}")

# ============================================
# 4. LOAD DATA
# ============================================

print("\n[Loading Data]")

# Data augmentation untuk training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_path = processed_dir / "train"
valid_path = processed_dir / "valid"
test_path = processed_dir / "test"

if not train_path.exists():
    print(f"❌ Data tidak ditemukan di: {train_path}")
    raise FileNotFoundError("Jalankan 02_Preprocessing.ipynb dulu!")

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
    class_mode='binary',
    classes=CONFIG['class_names'],
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    valid_path,
    target_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
    class_mode='binary',
    classes=CONFIG['class_names'],
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
    class_mode='binary',
    classes=CONFIG['class_names'],
    shuffle=False
)

print(f"\n   - Train samples: {train_generator.samples}")
print(f"   - Validation samples: {validation_generator.samples}")
print(f"   - Test samples: {test_generator.samples}")
print(f"   - Class indices: {train_generator.class_indices}")

# ============================================
# 5. BUILD MOBILENETV2
# ============================================

print("\n[Building MobileNetV2]")

# Load pre-trained MobileNetV2 (tanpa top layer)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

print(f"✅ Base model loaded: MobileNetV2")
print(f"   - Total layers: {len(base_model.layers)}")
print(f"   - Trainable: {base_model.trainable}")

# ============================================
# 6. PHASE 1: FEATURE EXTRACTION
# ============================================

print("\n" + "="*60)
print("🔒 PHASE 1: FEATURE EXTRACTION (Base Model Frozen)")
print("="*60)

# Freeze base model
base_model.trainable = False

# Build model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate']),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

print("✅ Model compiled (Phase 1)")
model.summary()

# Callbacks
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=CONFIG['early_stopping_patience'], 
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=CONFIG['reduce_lr_patience'],
                      min_lr=1e-7, verbose=1),
    ModelCheckpoint(model_dir / 'best_model_phase1.keras', monitor='val_accuracy', 
                    save_best_only=True, verbose=1),
    TensorBoard(log_dir=str(result_dir / f'tensorboard_phase1_{timestamp}'))
]

# Training Phase 1
print("\n[Training Phase 1]")
print("="*60)

start_time = time.time()

history_phase1 = model.fit(
    train_generator,
    epochs=CONFIG['phase1_epochs'],
    validation_data=validation_generator,
    callbacks=callbacks_phase1,
    verbose=1
)

phase1_time = time.time() - start_time
print(f"\n⏱️ Phase 1 time: {phase1_time/60:.2f} minutes")

# ============================================
# 7. PHASE 2: FINE-TUNING
# ============================================

print("\n" + "="*60)
print("🔓 PHASE 2: FINE-TUNING (Unfreeze Top Layers)")
print("="*60)

# Unfreeze sebagian layer
base_model.trainable = True

# Freeze layer awal (100 layer pertama tetap frozen)
for layer in base_model.layers[:100]:
    layer.trainable = False

# Cek layer yang trainable
trainable_layers = sum([1 for layer in base_model.layers if layer.trainable])
print(f"✅ Trainable layers: {trainable_layers} of {len(base_model.layers)}")
print(f"   - Layer 0-99: Frozen")
print(f"   - Layer 100-{len(base_model.layers)-1}: Trainable")

# Re-compile dengan learning rate lebih kecil
model.compile(
    optimizer=Adam(learning_rate=CONFIG['fine_tune_lr']),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

print("\n✅ Model re-compiled (Phase 2)")

# Callbacks Phase 2
callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=CONFIG['early_stopping_patience'], 
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=CONFIG['reduce_lr_patience'],
                      min_lr=1e-8, verbose=1),
    ModelCheckpoint(model_dir / 'best_model_phase2.keras', monitor='val_accuracy', 
                    save_best_only=True, verbose=1),
    TensorBoard(log_dir=str(result_dir / f'tensorboard_phase2_{timestamp}'))
]

# Training Phase 2
print("\n[Training Phase 2]")
print("="*60)

start_time = time.time()

history_phase2 = model.fit(
    train_generator,
    epochs=CONFIG['phase2_epochs'],
    validation_data=validation_generator,
    callbacks=callbacks_phase2,
    verbose=1
)

phase2_time = time.time() - start_time
print(f"\n⏱️ Phase 2 time: {phase2_time/60:.2f} minutes")

# Total time
total_time = phase1_time + phase2_time
print(f"\n⏱️ Total training time: {total_time/60:.2f} minutes")

# ============================================
# 8. SAVE FINAL MODEL
# ============================================

print("\n[Saving Model]")

# Save best model (Phase 2)
best_model = keras.models.load_model(model_dir / 'best_model_phase2.keras')
best_model.save(model_dir / 'best_model.keras')
model.save(model_dir / 'final_model.keras')

print(f"✅ Model saved to: {model_dir}")

# ============================================
# 9. EVALUASI
# ============================================

print("\n[Evaluating Model]")

# Test evaluation
test_loss, test_accuracy, test_precision, test_recall = best_model.evaluate(test_generator, verbose=0)

# Calculate F1
f1_score = 2 * (test_precision * test_recall) / (test_precision + test_recall)

print(f"\n📊 Test Set Results:")
print(f"   - Loss: {test_loss:.4f}")
print(f"   - Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   - Precision: {test_precision:.4f}")
print(f"   - Recall: {test_recall:.4f}")
print(f"   - F1-Score: {f1_score:.4f}")

# ============================================
# 10. PREDICTIONS & CONFUSION MATRIX
# ============================================

print("\n[Generating Predictions]")

test_generator.reset()
y_pred_proba = best_model.predict(test_generator, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
y_true = test_generator.classes

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot Confusion Matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CONFIG['class_names'],
            yticklabels=CONFIG['class_names'], ax=ax1)
ax1.set_title('Confusion Matrix - MobileNetV2', fontsize=14, fontweight='bold')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')

sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CONFIG['class_names'],
            yticklabels=CONFIG['class_names'], ax=ax2)
ax2.set_title('Confusion Matrix (Normalized) - MobileNetV2', fontsize=14, fontweight='bold')
ax2.set_ylabel('Actual')
ax2.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(result_dir / 'confusion_matrix.png', dpi=300)
plt.show()

# ============================================
# 11. CLASSIFICATION REPORT
# ============================================

print("\n[Classification Report]")

report = classification_report(y_true, y_pred, 
                               target_names=CONFIG['class_names'],
                               output_dict=True)

report_df = pd.DataFrame(report).transpose()
print("\n" + report_df.round(4).to_string())

# ============================================
# 12. ROC CURVE
# ============================================

print("\n[ROC Curve]")

fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, 
        label=f'MobileNetV2 (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - MobileNetV2', fontsize=14, fontweight='bold')
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(result_dir / 'roc_curve.png', dpi=300)
plt.show()

# ============================================
# 13. TRAINING HISTORY
# ============================================

print("\n[Training History]")

# Gabungkan history Phase 1 dan Phase 2
combined_history = {
    'accuracy': history_phase1.history['accuracy'] + history_phase2.history['accuracy'],
    'val_accuracy': history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy'],
    'loss': history_phase1.history['loss'] + history_phase2.history['loss'],
    'val_loss': history_phase1.history['val_loss'] + history_phase2.history['val_loss'],
    'precision': history_phase1.history['precision'] + history_phase2.history['precision'],
    'val_precision': history_phase1.history['val_precision'] + history_phase2.history['val_precision'],
    'recall': history_phase1.history['recall'] + history_phase2.history['recall'],
    'val_recall': history_phase1.history['val_recall'] + history_phase2.history['val_recall']
}

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy
axes[0, 0].plot(combined_history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(combined_history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 0].axvline(x=len(history_phase1.history['accuracy']), color='red', 
                   linestyle='--', linewidth=2, label='Fine-tuning start')
axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(combined_history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(combined_history['val_loss'], label='Validation', linewidth=2)
axes[0, 1].axvline(x=len(history_phase1.history['loss']), color='red', 
                   linestyle='--', linewidth=2, label='Fine-tuning start')
axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(combined_history['precision'], label='Train', linewidth=2)
axes[1, 0].plot(combined_history['val_precision'], label='Validation', linewidth=2)
axes[1, 0].axvline(x=len(history_phase1.history['precision']), color='red', 
                   linestyle='--', linewidth=2, label='Fine-tuning start')
axes[1, 0].set_title('Model Precision', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(combined_history['recall'], label='Train', linewidth=2)
axes[1, 1].plot(combined_history['val_recall'], label='Validation', linewidth=2)
axes[1, 1].axvline(x=len(history_phase1.history['recall']), color='red', 
                   linestyle='--', linewidth=2, label='Fine-tuning start')
axes[1, 1].set_title('Model Recall', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(result_dir / 'training_history.png', dpi=300)
plt.show()

# ============================================
# 14. EXPORT RESULTS
# ============================================

print("\n[Exporting Results]")

metrics = {
    'model': 'MobileNetV2 (Transfer Learning + Fine-tuning)',
    'timestamp': timestamp,
    'phase1_epochs': CONFIG['phase1_epochs'],
    'phase2_epochs': CONFIG['phase2_epochs'],
    'total_epochs': len(combined_history['accuracy']),
    'phase1_time_minutes': phase1_time/60,
    'phase2_time_minutes': phase2_time/60,
    'total_time_minutes': total_time/60,
    'test_metrics': {
        'loss': float(test_loss),
        'accuracy': float(test_accuracy),
        'precision': float(test_precision),
        'recall': float(test_recall),
        'f1_score': float(f1_score),
        'auc_roc': float(roc_auc)
    },
    'best_validation': {
        'accuracy': float(max(combined_history['val_accuracy'])),
        'loss': float(min(combined_history['val_loss']))
    },
    'config': CONFIG,
    'confusion_matrix': cm.tolist(),
    'classification_report': report
}

with open(result_dir / 'results.json', 'w') as f:
    json.dump(metrics, f, indent=4)

# Export training history
history_export = {
    'epochs': list(range(1, len(combined_history['accuracy']) + 1)),
    'accuracy': combined_history['accuracy'],
    'val_accuracy': combined_history['val_accuracy'],
    'loss': combined_history['loss'],
    'val_loss': combined_history['val_loss'],
    'precision': combined_history['precision'],
    'val_precision': combined_history['val_precision'],
    'recall': combined_history['recall'],
    'val_recall': combined_history['val_recall']
}

with open(result_dir / 'training_history.json', 'w') as f:
    json.dump(history_export, f, indent=4)

# Export classification report
report_df.to_csv(result_dir / 'classification_report.csv')

# Export confusion matrix
np.savetxt(result_dir / 'confusion_matrix.csv', cm, delimiter=',', fmt='%d')

print(f"✅ All results exported to: {result_dir}")

# ============================================
# 15. RINGKASAN AKHIR
# ============================================

print("\n" + "="*60)
print("✅ MOBILENETV2 TRAINING COMPLETE!")
print("="*60)

print(f"\n📊 RINGKASAN HASIL:")
print(f"   - Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   - Test Precision: {test_precision:.4f}")
print(f"   - Test Recall: {test_recall:.4f}")
print(f"   - Test F1-Score: {f1_score:.4f}")
print(f"   - AUC-ROC: {roc_auc:.4f}")
print(f"   - Training Time: {total_time/60:.2f} minutes")

print(f"\n📁 Hasil disimpan di: {result_dir}")
print(f"   - best_model.keras: {model_dir / 'best_model.keras'}")
print(f"   - results.json: {result_dir / 'results.json'}")
print(f"   - training_history.json: {result_dir / 'training_history.json'}")
print(f"   - confusion_matrix.png: {result_dir / 'confusion_matrix.png'}")
print(f"   - roc_curve.png: {result_dir / 'roc_curve.png'}")

print("\n" + "="*60)
print("🚀 LANJUT KE YOLOv8 TRAINING (05_YOLOv8.ipynb)")
print("="*60)